# 02 — Full Baseline Evaluation trên Golden Test Set

**Mục tiêu:** Đo accuracy của Gemini với **prompt naive** (không có role, CoT, few-shot, hay Pydantic schema) trên **toàn bộ 100 ảnh golden test set**.

Kết quả notebook này là **điểm xuất phát** để so sánh với mọi cải tiến sau:
- Prompt v1 (có role + CoT + few-shot) → notebook 03
- Preprocessing + Prompt v1 → notebook 04

**Prompt strategy:** Hỏi Gemini trả về JSON tối giản — không có schema enforcement, không có Pydantic, không có structured output API. Đây vẫn là "naive" vì:
- Không có role definition
- Không có chain-of-thought
- Không có few-shot examples
- Không có self-check instruction
- Không có explicit anti-hallucination rules

| Metric target (baseline) | Expected |
|---|---|
| End-to-end success | ~50–65% |
| Field-level (merchant) | ~60–75% |
| Field-level (date) | ~65–80% |
| Field-level (total) | ~70–80% |
| Category accuracy | ~60–70% |
| Items F1 | ~40–60% |

In [1]:
# Cell 1 — Imports
import io
import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path

import PIL.Image
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, display

# Đảm bảo src/ có thể import được từ notebook
_cwd = Path.cwd()
PROJECT_ROOT = _cwd if (_cwd / ".env").exists() else _cwd.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import evaluate, get_failures

print("Imports OK")
print("Project root:", PROJECT_ROOT.resolve())

Imports OK
Project root: C:\LamHoangPhuc\AI\Smart_Receipt_VN


In [2]:
# Cell 2 — Config & paths
load_dotenv(dotenv_path=PROJECT_ROOT / ".env")
api_key = os.getenv("Gemini_API_Key")
assert api_key, "Không tìm thấy Gemini_API_Key trong .env!"

MODEL        = "gemini-2.5-flash-lite"
GOLDEN_PATH  = PROJECT_ROOT / "data" / "golden_test_set.json"
RAW_DIR      = PROJECT_ROOT / "data" / "raw"
CHECKPOINT   = PROJECT_ROOT / "data" / "baseline_eval_checkpoint.json"  # resume nếu bị ngắt
EXP_OUT      = PROJECT_ROOT / "docs" / "experiments" / "exp_001_naive_baseline.md"

MAX_IMG_WIDTH  = 900    # px — giảm token cost
CALL_DELAY_SEC = 4.5    # giây giữa các call (free tier: 15 RPM)
MAX_RETRIES    = 4

client = genai.Client(api_key=api_key)
print("Model:", MODEL)
print("Golden set:", GOLDEN_PATH)
print(f"Delay between calls: {CALL_DELAY_SEC}s → ~{100*CALL_DELAY_SEC/60:.0f} phút cho 100 ảnh")

Model: gemini-2.5-flash-lite
Golden set: c:\LamHoangPhuc\AI\Smart_Receipt_VN\data\golden_test_set.json
Delay between calls: 4.5s → ~8 phút cho 100 ảnh


In [3]:
# Cell 3 — Load golden test set
with open(GOLDEN_PATH, encoding="utf-8") as f:
    golden = json.load(f)

print(f"Golden test set: {len(golden)} samples")

from collections import Counter
group_dist = Counter(x["group"] for x in golden)
for g, n in sorted(group_dist.items()):
    print(f"  {g}: {n}")

Golden test set: 100 samples
  01_pos_clean: 26
  02_vat_invoice: 17
  03_long_invoice: 16
  04_handwritten: 11
  05_low_quality: 30


In [5]:
# Cell 4 — Naive baseline prompt
#
# NAIVE = không có:
#   - Role definition ("You are a Vietnamese accounting expert...")
#   - Chain-of-thought steps
#   - Few-shot examples
#   - Self-check / anti-hallucination rules
#   - Pydantic schema / structured output API
#
# Chỉ yêu cầu JSON tối giản để có thể parse và đo bằng metrics.py.

CATEGORIES = [
    "Ăn uống", "Đi lại", "Mua sắm", "Giải trí",
    "Hoá đơn tiện ích", "Sức khoẻ", "Giáo dục",
    "Du lịch", "Nhà cửa", "Khác",
]

NAIVE_PROMPT = """Extract information from this receipt image and return a JSON object.

Return only a JSON object with these fields:
- merchant_name: string or null
- date: string in DD-MM-YYYY format, or null
- total_amount: integer (VND, no commas), or null
- items: list of {"name": string, "quantity": number, "unit_price": integer, "total": integer}
- category: one of """ + json.dumps(CATEGORIES, ensure_ascii=False) + """

Return only the JSON, no explanation."""

print("Prompt (naive baseline):")
print("-" * 60)
print(NAIVE_PROMPT)
print("-" * 60)
print(f"\nLength: {len(NAIVE_PROMPT)} chars")

Prompt (naive baseline):
------------------------------------------------------------
Extract information from this receipt image and return a JSON object.

Return only a JSON object with these fields:
- merchant_name: string or null
- date: string in DD-MM-YYYY format, or null
- total_amount: integer (VND, no commas), or null
- items: list of {"name": string, "quantity": number, "unit_price": integer, "total": integer}
- category: one of ["Ăn uống", "Đi lại", "Mua sắm", "Giải trí", "Hoá đơn tiện ích", "Sức khoẻ", "Giáo dục", "Du lịch", "Nhà cửa", "Khác"]

Return only the JSON, no explanation.
------------------------------------------------------------

Length: 514 chars


In [6]:
# Cell 5 — Helper functions

def to_jpeg_bytes(img: PIL.Image.Image, max_width: int = MAX_IMG_WIDTH) -> io.BytesIO:
    """Resize nếu quá rộng, convert sang JPEG bytes."""
    w, h = img.size
    if w > max_width:
        img = img.resize((max_width, int(h * max_width / w)))
    if img.mode not in ("RGB", "L"):
        img = img.convert("RGB")
    buf = io.BytesIO()
    img.save(buf, format="JPEG", quality=85)
    buf.seek(0)
    return buf


def call_gemini(img_bytes: io.BytesIO, prompt: str) -> tuple[str, str, str]:
    """Gọi Gemini với retry. Returns (text, status, error)."""
    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(
                model=MODEL,
                contents=[
                    types.Part.from_bytes(data=img_bytes.getvalue(), mime_type="image/jpeg"),
                    prompt,
                ],
            )
            return response.text, "ok", ""
        except Exception as e:
            err = str(e)
            retryable = any(x in err for x in ["429", "RESOURCE_EXHAUSTED", "503", "UNAVAILABLE", "overloaded"])
            if retryable and attempt < MAX_RETRIES - 1:
                wait = 5 * (2 ** attempt)
                print(f"  [retry {attempt+1}] chờ {wait}s...", end=" ", flush=True)
                time.sleep(wait)
            else:
                return "", "error", err[:200]
    return "", "error", "max retries exceeded"


def parse_response(raw: str) -> dict | None:
    """
    Extract JSON từ response Gemini.
    Xử lý cả trường hợp Gemini wrap trong ```json ... ``` hoặc trả thẳng JSON.
    Trả về None nếu không parse được.
    """
    if not raw:
        return None
    # Thử extract từ code block trước
    match = re.search(r"```(?:json)?\s*([\s\S]+?)```", raw)
    candidate = match.group(1).strip() if match else raw.strip()
    # Fallback: tìm { ... } lớn nhất
    if not candidate.startswith("{"):
        brace = re.search(r"(\{[\s\S]+\})", raw)
        candidate = brace.group(1) if brace else candidate
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return None


print("Helper functions defined.")

Helper functions defined.


In [7]:
# Cell 6 — Chạy Gemini trên 100 ảnh (với checkpoint để resume nếu bị ngắt)
#
# Checkpoint được lưu sau MỖI ảnh → có thể Interrupt kernel và chạy tiếp từ đây.
# Xoá file checkpoint nếu muốn chạy lại từ đầu.

# Load checkpoint nếu có
raw_results: dict[str, dict] = {}   # image → {raw, status, error, latency_sec}
if CHECKPOINT.exists():
    with open(CHECKPOINT, encoding="utf-8") as f:
        raw_results = json.load(f)
    print(f"Checkpoint loaded: {len(raw_results)} ảnh đã có kết quả")
else:
    print("Không có checkpoint — bắt đầu từ đầu")

# Xác định ảnh còn lại cần xử lý
todo = [g for g in golden if g["image"] not in raw_results]
print(f"Còn lại: {len(todo)} ảnh cần gửi API")
if not todo:
    print("Tất cả ảnh đã có kết quả. Chạy cell tiếp theo để parse.")

# --- Gọi API ---
for i, record in enumerate(todo, 1):
    img_path = RAW_DIR / record["group"] / record["image"]
    print(f"[{i}/{len(todo)}] {record['group']}/{record['image']} ... ", end="", flush=True)

    start = time.perf_counter()
    try:
        img = PIL.Image.open(img_path)
        img_bytes = to_jpeg_bytes(img)
        raw_text, status, error = call_gemini(img_bytes, NAIVE_PROMPT)
    except Exception as e:
        raw_text, status, error = "", "error", str(e)[:200]

    elapsed = time.perf_counter() - start
    print(f"{status} ({elapsed:.1f}s)")

    raw_results[record["image"]] = {
        "raw": raw_text,
        "status": status,
        "error": error,
        "latency_sec": elapsed,
    }

    # Lưu checkpoint sau mỗi ảnh
    with open(CHECKPOINT, "w", encoding="utf-8") as f:
        json.dump(raw_results, f, ensure_ascii=False, indent=2)

    # Delay để không vượt rate limit
    if i < len(todo):
        time.sleep(CALL_DELAY_SEC)

ok_count = sum(1 for v in raw_results.values() if v["status"] == "ok")
print(f"\nHoàn thành: {ok_count}/{len(golden)} ảnh OK")

Không có checkpoint — bắt đầu từ đầu
Còn lại: 100 ảnh cần gửi API
[1/100] 02_vat_invoice/vat_invoice_0001.jpg ... ok (5.4s)
[2/100] 02_vat_invoice/vat_invoice_0028.jpg ... ok (4.8s)
[3/100] 05_low_quality/low_quality_0011.jpg ... ok (2.2s)
[4/100] 01_pos_clean/pos_clean_0003.jpg ... ok (2.5s)
[5/100] 04_handwritten/handwritten_0007.jpg ... ok (2.3s)
[6/100] 03_long_invoice/long_invoice_0012.jpg ... ok (6.8s)
[7/100] 01_pos_clean/pos_clean_0024.jpg ... ok (2.6s)
[8/100] 05_low_quality/low_quality_0040.jpg ... ok (2.3s)
[9/100] 01_pos_clean/pos_clean_0035.jpg ... ok (2.8s)
[10/100] 05_low_quality/low_quality_0036.jpg ... ok (3.8s)
[11/100] 05_low_quality/low_quality_0012.jpg ... ok (2.7s)
[12/100] 01_pos_clean/pos_clean_0040.jpg ... ok (2.7s)
[13/100] 03_long_invoice/long_invoice_0020.jpg ... ok (3.5s)
[14/100] 03_long_invoice/long_invoice_0032.jpg ... ok (5.6s)
[15/100] 05_low_quality/low_quality_0058.jpg ... ok (2.1s)
[16/100] 03_long_invoice/long_invoice_0010.jpg ... ok (3.9s)
[17/100

In [8]:
# Cell 7 — Parse responses → prediction dicts
#
# Mỗi prediction dict phải có cùng schema với golden test set.
# Nếu parse fail → dùng empty dict (tất cả fields None → evaluate sẽ coi là fail).

predictions = []
latencies   = []
parse_errors = []

for record in golden:
    result = raw_results.get(record["image"], {})
    latencies.append(result.get("latency_sec", 0.0))

    if result.get("status") != "ok":
        # API error → empty prediction
        predictions.append({"image": record["image"], "group": record["group"]})
        parse_errors.append({"image": record["image"], "reason": f"api_error: {result.get('error', '')[:80]}"})
        continue

    parsed = parse_response(result.get("raw", ""))
    if parsed is None:
        predictions.append({"image": record["image"], "group": record["group"]})
        parse_errors.append({"image": record["image"], "reason": "json_parse_failed",
                              "raw_preview": result.get("raw", "")[:120]})
    else:
        # Merge metadata fields vào prediction để evaluate_sample biết image/group
        parsed["image"] = record["image"]
        parsed["group"] = record["group"]
        predictions.append(parsed)

print(f"Parsed OK : {len(predictions) - len(parse_errors)}/{len(golden)}")
print(f"Parse fail: {len(parse_errors)}")
if parse_errors:
    print("\nParse failures:")
    for e in parse_errors:
        print(f"  {e['image']}: {e['reason']}")
        if "raw_preview" in e:
            print(f"    preview: {e['raw_preview']}")

Parsed OK : 100/100
Parse fail: 0


In [9]:
# Cell 8 — Chạy evaluate() → kết quả tổng hợp

report = evaluate(predictions, golden, latencies=latencies)

display(Markdown("## Kết quả Baseline — Gemini naive prompt"))
display(Markdown(f"""
| Metric | Score |
|---|---|
| **End-to-end success** | **{report['end_to_end_success']:.1%}** |
| Merchant accuracy | {report['field_accuracy']['merchant']:.1%} |
| Date accuracy | {report['field_accuracy']['date']:.1%} |
| Total accuracy | {report['field_accuracy']['total']:.1%} |
| Items F1 | {report['items_f1']:.1%} |
| Items Precision | {report['items_precision']:.1%} |
| Items Recall | {report['items_recall']:.1%} |
| Category accuracy | {report['category_accuracy']:.1%} |
| Latency p50 | {report['latency_p50']:.2f}s |
| Latency p95 | {report['latency_p95']:.2f}s |
| n samples | {report['n_samples']} |
"""))

## Kết quả Baseline — Gemini naive prompt


| Metric | Score |
|---|---|
| **End-to-end success** | **62.0%** |
| Merchant accuracy | 78.0% |
| Date accuracy | 81.0% |
| Total accuracy | 92.0% |
| Items F1 | 85.3% |
| Items Precision | 89.5% |
| Items Recall | 85.7% |
| Category accuracy | 75.0% |
| Latency p50 | 3.22s |
| Latency p95 | 6.44s |
| n samples | 100 |


In [10]:
# Cell 9 — Per-group breakdown

group_labels = {
    "01_pos_clean"   : "POS rõ nét",
    "02_vat_invoice" : "VAT điện tử",
    "03_long_invoice": "Hoá đơn dài",
    "04_handwritten" : "Viết tay",
    "05_low_quality" : "Chất lượng kém",
}

rows = ""
for g, stats in report["per_group"].items():
    label = group_labels.get(g, g)
    rows += (
        f"| {g} ({label}) "
        f"| {stats['n']} "
        f"| {stats['end_to_end_success']:.1%} "
        f"| {stats['merchant_accuracy']:.1%} "
        f"| {stats['date_accuracy']:.1%} "
        f"| {stats['total_accuracy']:.1%} "
        f"| {stats['items_f1']:.1%} "
        f"| {stats['category_accuracy']:.1%} |\n"
    )

display(Markdown("## Per-group breakdown"))
display(Markdown(
    "| Nhóm | n | E2E | Merchant | Date | Total | Items F1 | Category |\n"
    "|---|---|---|---|---|---|---|---|\n" + rows
))

## Per-group breakdown

| Nhóm | n | E2E | Merchant | Date | Total | Items F1 | Category |
|---|---|---|---|---|---|---|---|
| 01_pos_clean (POS rõ nét) | 26 | 73.1% | 88.5% | 88.5% | 96.2% | 75.9% | 88.5% |
| 02_vat_invoice (VAT điện tử) | 17 | 58.8% | 76.5% | 82.4% | 100.0% | 98.0% | 82.4% |
| 03_long_invoice (Hoá đơn dài) | 16 | 75.0% | 87.5% | 100.0% | 87.5% | 98.1% | 50.0% |
| 04_handwritten (Viết tay) | 11 | 27.3% | 54.5% | 45.5% | 54.5% | 77.0% | 63.6% |
| 05_low_quality (Chất lượng kém) | 30 | 60.0% | 73.3% | 76.7% | 100.0% | 82.5% | 76.7% |


In [11]:
# Cell 10 — Per-category accuracy

rows = ""
for cat, acc in sorted(report["per_category_accuracy"].items(), key=lambda x: -x[1]):
    n_cat = sum(1 for g in golden if g.get("category") == cat)
    rows += f"| {cat} | {n_cat} | {acc:.1%} |\n"

display(Markdown("## Per-category accuracy"))
display(Markdown(
    "| Category | n | Accuracy |\n"
    "|---|---|---|\n" + rows
))

## Per-category accuracy

| Category | n | Accuracy |
|---|---|---|
| Giải trí | 2 | 100.0% |
| Hoá đơn tiện ích | 5 | 100.0% |
| Sức khoẻ | 3 | 100.0% |
| Đi lại | 15 | 93.3% |
| Mua sắm | 10 | 80.0% |
| Nhà cửa | 5 | 80.0% |
| Giáo dục | 7 | 71.4% |
| Ăn uống | 41 | 65.9% |
| Du lịch | 5 | 60.0% |
| Khác | 7 | 57.1% |


In [12]:
# Cell 11 — Failure analysis (top 20 worst cases)
#
# Mục đích: xác định pattern lỗi để viết prompt v1 tốt hơn.

failures = get_failures(predictions, golden, latencies=latencies)

display(Markdown(f"## Failure Analysis ({len(failures)}/{len(golden)} samples fail end-to-end)"))

# Thống kê theo loại lỗi
only_merchant_fail = [f for f in failures if not f["merchant_ok"] and f["date_ok"] and f["total_ok"]]
only_date_fail     = [f for f in failures if f["merchant_ok"] and not f["date_ok"] and f["total_ok"]]
only_total_fail    = [f for f in failures if f["merchant_ok"] and f["date_ok"] and not f["total_ok"]]
multi_fail         = [f for f in failures if sum([not f["merchant_ok"], not f["date_ok"], not f["total_ok"]]) > 1]

display(Markdown(f"""
| Loại lỗi | Count |
|---|---|
| Chỉ merchant sai | {len(only_merchant_fail)} |
| Chỉ date sai | {len(only_date_fail)} |
| Chỉ total sai | {len(only_total_fail)} |
| Nhiều fields sai | {len(multi_fail)} |
"""))

# Hiển thị 20 failure đầu
display(Markdown("### Chi tiết 20 failures đầu"))
rows = ""
for f in failures[:20]:
    merchant_flag = "✗" if not f["merchant_ok"] else "✓"
    date_flag     = "✗" if not f["date_ok"]     else "✓"
    total_flag    = "✗" if not f["total_ok"]    else "✓"
    rows += (
        f"| `{f['image']}` | {f['group'].split('_')[0]} "
        f"| {merchant_flag} `{str(f.get('pred_merchant',''))[:20]}` → `{str(f.get('gt_merchant',''))[:20]}` "
        f"| {date_flag} `{f.get('pred_date','')}` → `{f.get('gt_date','')}` "
        f"| {total_flag} `{f.get('pred_total','')}` → `{f.get('gt_total','')}` "
        f"| {f['items_f1']:.0%} |\n"
    )
display(Markdown(
    "| Image | Group | Merchant | Date | Total | Items F1 |\n"
    "|---|---|---|---|---|---|\n" + rows
))

## Failure Analysis (38/100 samples fail end-to-end)


| Loại lỗi | Count |
|---|---|
| Chỉ merchant sai | 14 |
| Chỉ date sai | 12 |
| Chỉ total sai | 4 |
| Nhiều fields sai | 8 |


### Chi tiết 20 failures đầu

| Image | Group | Merchant | Date | Total | Items F1 |
|---|---|---|---|---|---|
| `vat_invoice_0001.jpg` | 02 | ✓ `CÔNG TY CỔ PHẦN ĐẦU ` → `CÔNG TY CỔ PHẦN ĐẦU ` | ✗ `06-2026` → `06-03-2026` | ✓ `2074800` → `2074800` | 100% |
| `handwritten_0007.jpg` | 04 | ✗ `VINASUN` → `CTY CP ÁNH DƯƠNG VN` | ✗ `None` → `26-11-2020` | ✗ `100000` → `150000` | 0% |
| `pos_clean_0024.jpg` | 01 | ✗ `CÔNG TY XNK THANH HÀ` → `KHÁCH SẠN TRE XANH` | ✓ `01-06-2025` → `01-06-2025` | ✓ `8000000` → `8000000` | 100% |
| `low_quality_0042.jpg` | 05 | ✗ `Trà Sen` → `trasencodau` | ✓ `24-10-2024` → `24-10-2024` | ✓ `60000` → `60000` | 100% |
| `vat_invoice_0023.jpg` | 02 | ✗ `GM THANG LONG` → `CÔNG TY CỔ PHẦN GM T` | ✓ `30-09-2016` → `30-09-2016` | ✓ `609000000` → `609000000` | 100% |
| `low_quality_0000.jpg` | 05 | ✗ `BIDV` → `NGÂN HÀNG TMCP ĐẦU T` | ✗ `23-09-2016` → `16-09-2023` | ✓ `1055000` → `1055000` | 50% |
| `long_invoice_0005.jpg` | 03 | ✗ `Co.op Food HN THE K-` → `Saigon Co.op` | ✓ `20-09-2019` → `20-09-2019` | ✓ `27291` → `27291` | 100% |
| `low_quality_0038.jpg` | 05 | ✗ `None` → `null` | ✗ `14-11-2016` → `14-01-2016` | ✓ `74000` → `74000` | 50% |
| `vat_invoice_0020.jpg` | 02 | ✗ `XANH SM` → `CHI NHÁNH THÀNH PHỐ ` | ✓ `27-09-2025` → `27-09-2025` | ✓ `250000` → `250000` | 100% |
| `pos_clean_0012.jpg` | 01 | ✓ `None` → `None` | ✗ `06-10-2025` → `10-06-2025` | ✓ `10500000` → `10500000` | 100% |
| `handwritten_0000.jpg` | 04 | ✗ `TUẤN DUNG TRANSLATIO` → `CÔNG TY DỊCH THUẬT T` | ✗ `26-01-2017` → `26-04-2017` | ✓ `250000` → `250000` | 100% |
| `handwritten_0004.jpg` | 04 | ✓ `IALC VIỆT NAM - CN T` → `TRUNG TÂM ANH NGỮ IA` | ✗ `28-08-2025` → `02-08-2025` | ✓ `9000000` → `9000000` | 100% |
| `long_invoice_0015.jpg` | 03 | ✗ `Co.op Food HN THE K-` → `Saigon Co.op` | ✓ `20-09-2019` → `20-09-2019` | ✓ `38774` → `38774` | 100% |
| `low_quality_0027.jpg` | 05 | ✓ `KFC` → `Cty LD TNHH KFC Viet` | ✗ `15-08-2019` → `15-08-2020` | ✓ `51000` → `51000` | 100% |
| `low_quality_0046.jpg` | 05 | ✗ `Petrolimex` → `Công ty xăng dầu Điệ` | ✓ `30-03-2018` → `30-03-2018` | ✓ `1860000` → `1860000` | 100% |
| `vat_invoice_0009.jpg` | 02 | ✗ `VIETTEL` → `TẬP ĐOÀN CÔNG NGHIỆP` | ✓ `02-04-2024` → `02-04-2024` | ✓ `177993000` → `177993000` | 100% |
| `pos_clean_0026.jpg` | 01 | ✓ `Bảo Trang` → `Nhà hàng Bảo Trang (` | ✗ `02-08-2017` → `08-02-2017` | ✓ `1344600` → `1344600` | 100% |
| `vat_invoice_0022.jpg` | 02 | ✓ `CÔNG TY TNHH MỘT THÀ` → `CÔNG TY TNHH MỘT THÀ` | ✗ `07-07-2022` → `08-07-2022` | ✓ `123760` → `123760` | 100% |
| `handwritten_0002.jpg` | 04 | ✓ `NGUYỄN HOÀI NAM` → `NGUYỄN HOÀI NAM` | ✗ `27-05-2008` → `27-08-2008` | ✓ `500000` → `500000` | 100% |
| `handwritten_0001.jpg` | 04 | ✗ `NHÀ HÀNG DÂN` → `NHẬU BÌNH DÂN CÁC MÓ` | ✓ `None` → `None` | ✗ `370` → `370000` | 67% |


In [13]:
# Cell 12 — Lưu experiment document
#
# Tạo docs/experiments/exp_001_naive_baseline.md với đầy đủ metrics.
# File này là ground truth để so sánh với mọi experiment sau.

EXP_OUT.parent.mkdir(parents=True, exist_ok=True)

run_ts = datetime.now().strftime("%Y-%m-%d %H:%M")

# Per-group table
group_rows = ""
for g, stats in report["per_group"].items():
    group_rows += (
        f"| {g} | {stats['n']} "
        f"| {stats['end_to_end_success']:.1%} "
        f"| {stats['merchant_accuracy']:.1%} "
        f"| {stats['date_accuracy']:.1%} "
        f"| {stats['total_accuracy']:.1%} "
        f"| {stats['items_f1']:.1%} "
        f"| {stats['category_accuracy']:.1%} |\n"
    )

# Per-category table
cat_rows = ""
for cat, acc in sorted(report["per_category_accuracy"].items(), key=lambda x: -x[1]):
    n_cat = sum(1 for g in golden if g.get("category") == cat)
    cat_rows += f"| {cat} | {n_cat} | {acc:.1%} |\n"

# Failure pattern summary
failures_all = get_failures(predictions, golden)
only_m = sum(1 for f in failures_all if not f["merchant_ok"] and f["date_ok"] and f["total_ok"])
only_d = sum(1 for f in failures_all if f["merchant_ok"] and not f["date_ok"] and f["total_ok"])
only_t = sum(1 for f in failures_all if f["merchant_ok"] and f["date_ok"] and not f["total_ok"])
multi  = sum(1 for f in failures_all if sum([not f["merchant_ok"], not f["date_ok"], not f["total_ok"]]) > 1)

doc = f"""# exp_001 — Gemini Naive Baseline

**Date:** {run_ts}  
**Model:** {MODEL}  
**Prompt version:** naive_v0 (no role, no CoT, no few-shot, no structured output API)  
**Dataset:** golden_test_set.json ({report['n_samples']} samples)

## Hypothesis

Gemini Vision với prompt tối giản (chỉ yêu cầu JSON, không có kỹ thuật prompting) sẽ cho kết quả ở mức ~50–65% end-to-end success — đủ để prove concept nhưng rõ ràng cần cải thiện.

## Prompt

```
{NAIVE_PROMPT}
```

## Results

### Overall

| Metric | Score |
|---|---|
| **End-to-end success** | **{report['end_to_end_success']:.1%}** |
| Merchant accuracy | {report['field_accuracy']['merchant']:.1%} |
| Date accuracy | {report['field_accuracy']['date']:.1%} |
| Total accuracy | {report['field_accuracy']['total']:.1%} |
| Items F1 | {report['items_f1']:.1%} |
| Items Precision | {report['items_precision']:.1%} |
| Items Recall | {report['items_recall']:.1%} |
| Category accuracy | {report['category_accuracy']:.1%} |
| Latency p50 | {report['latency_p50']:.2f}s |
| Latency p95 | {report['latency_p95']:.2f}s |

### Per-group

| Group | n | E2E | Merchant | Date | Total | Items F1 | Category |
|---|---|---|---|---|---|---|---|
{group_rows}
### Per-category

| Category | n | Accuracy |
|---|---|---|
{cat_rows}
## Failure Analysis

**Total failures:** {len(failures_all)}/{report['n_samples']} ({len(failures_all)/report['n_samples']:.1%})

| Failure type | Count |
|---|---|
| Chỉ merchant sai | {only_m} |
| Chỉ date sai | {only_d} |
| Chỉ total sai | {only_t} |
| Nhiều fields sai | {multi} |

## Observations

*(Điền sau khi đọc qua failure cases)*

- [ ] TODO: Gemini có bỏ dấu tiếng Việt không?
- [ ] TODO: Format date có nhất quán không (DD-MM-YYYY vs YYYY-MM-DD)?
- [ ] TODO: Nhóm nào fail nhiều nhất?
- [ ] TODO: Category confusion matrix — hay nhầm giữa nhóm nào?

## Decision

Baseline thiết lập. Các bước tiếp theo (prompt v1, notebook 03):
1. Thêm role definition — "You are a Vietnamese accounting expert..."
2. Thêm chain-of-thought steps
3. Thêm few-shot examples (2-3 hard cases)
4. Thêm anti-hallucination constraints
5. Dùng structured output API (Pydantic schema)
6. Benchmark v1 vs naive_v0 trên cùng golden set
"""

with open(EXP_OUT, "w", encoding="utf-8") as f:
    f.write(doc)

print(f"Saved → {EXP_OUT.resolve()}")
display(Markdown(f"**Experiment document saved:** `{EXP_OUT}`"))

Saved → C:\LamHoangPhuc\AI\Smart_Receipt_VN\docs\experiments\exp_001_naive_baseline.md


**Experiment document saved:** `c:\LamHoangPhuc\AI\Smart_Receipt_VN\docs\experiments\exp_001_naive_baseline.md`